In [6]:
from langgraph.graph import StateGraph , START , END
from langchain_mistralai import ChatMistralAI
from typing import TypedDict , Literal , Annotated
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage , HumanMessage
from pydantic import BaseModel , Field

In [2]:
load_dotenv()

True

In [3]:
generate_llm = ChatMistralAI(
    model_name="mistral-small-latest",
    temperature=0
)

evaluate_llm = ChatMistralAI(
    model_name="mistral-large-latest",
    temperature=0
)

optimize_llm = ChatMistralAI(
    model_name="mistral-large-latest",
    temperature=0.3
)

In [7]:
class postEvaluation(BaseModel):
    evaluation: Literal["approved" , "needs_improvement"] = Field(description="Final evaluation result.")
    feedback: str = Field(description="Feedback for post")


In [4]:
class linkedinState(TypedDict):
    
    topic: str
    post: str
    evaluation: Literal["approved" , "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [8]:
structured_evaluator_llm = evaluate_llm.with_structured_output(linkedinState)

In [5]:
def generate_post(state: linkedinState):
    
    # prompt

    messages = [
    SystemMessage(content="You are a professional LinkedIn content creator who writes insightful, engaging, and authentic posts."),
    
    HumanMessage(content=f"""
    Write an original and engaging LinkedIn post on the topic: "{state['topic']}".

    Rules:
    - Do NOT use question-answer format.
    - Keep the tone professional, authentic, and human.
    - Start with a strong hook that grabs attention.
    - Share a clear insight, lesson, experience, or practical takeaway.
    - Use simple, easy-to-understand English.
    - Avoid generic corporate jargon and overly formal language.
    - Make the post relatable and valuable to LinkedIn professionals.
    - Use short paragraphs for readability.
    - Use emojis sparingly and only when they add value.
    - Include relevant hashtags at the end.
    - Do NOT use clickbait.
    - Do NOT sound like an AI-generated post.
    - Keep the post concise and engaging.
    - Do not repeat the topic unnecessarily.

    Write only the final LinkedIn post.
    """)
    ]
    
    response = generate_llm.invoke(messages).content
    
    return {"post": response}

    

In [9]:
def evaluate_post(state: linkedinState):
    # prompt
    messages = [
    SystemMessage(content="You are a ruthless but fair LinkedIn content critic. You evaluate LinkedIn posts based on professionalism, originality, engagement, clarity, value, authenticity, and LinkedIn-friendly formatting."),
    
    HumanMessage(content=f"""
    Evaluate the following LinkedIn post:

    Post: "{state['post']}"

    Use the criteria below to evaluate the post:

    1. Originality – Does the post offer a fresh perspective, personal insight, or unique take rather than generic LinkedIn content?
    2. Hook – Does the opening line grab attention and encourage the reader to continue?
    3. Value – Does the post provide a useful insight, lesson, experience, or takeaway?
    4. Engagement Potential – Is it likely to encourage meaningful reactions, comments, or shares?
    5. Authenticity – Does it sound human, natural, and genuine rather than AI-generated or overly corporate?
    6. Clarity – Is the message easy to understand with simple, readable English?
    7. Professionalism – Is the tone appropriate for LinkedIn without being unnecessarily formal?
    8. Structure & Readability – Does it use short paragraphs, whitespace, and a logical flow?
    9. Hashtags – Are the hashtags relevant and limited to a reasonable number?
    10. Overall Impact – Does the post leave the reader with a memorable insight or takeaway?

    Auto-reject if:
    - It sounds generic, robotic, or obviously AI-generated.
    - It uses excessive corporate jargon or buzzwords.
    - It starts with a weak or generic hook.
    - It provides little or no useful value.
    - It contains exaggerated, misleading, or unsupported claims.
    - It uses excessive emojis or hashtags.
    - It is written in question-answer format.
    - It uses clickbait or engagement bait.
    - It contains repetitive or unnecessary content.
    - It ends with a generic, forced, or meaningless conclusion.
    - It feels like a promotional advertisement rather than a valuable LinkedIn post.

    ### Respond ONLY in structured format:
    - evaluation: "approved" or "needs_improvement"
    - feedback: One paragraph explaining the strengths and weaknesses
    """)
    ]
    
    response = structured_evaluator_llm.invoke(messages)
    
    return {"evaluation": response.evaluation , "feedback": response.feedback}

In [ ]:
graph = StateGraph(linkedinState)

graph.add_node("generate" , generate_post)
graph.add_node("evaluate" , evaluate_post)
graph.add_node("optimize" , optimize_post)